In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np

import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# 测试torch版本和cuda是否安装成功
print(torch.__version__)
print(torch.cuda.is_available())

# 测试torch的功能
x = torch.rand(5, 3)
print(x)

2.8.0+cpu
False
tensor([[0.5227, 0.5718, 0.3363],
        [0.4785, 0.0688, 0.4981],
        [0.0194, 0.4598, 0.1502],
        [0.8961, 0.4494, 0.8056],
        [0.8996, 0.7073, 0.2239]])


### 数据预处理部分

In [2]:
import pandas as pd

movies = pd.read_csv('./movies.dat', sep='::', engine='python', names=['movie_id', 'title', 'genres'], encoding='latin-1')
ratings = pd.read_csv('./ratings.dat', sep='::', engine='python', names=['user_id', 'movie_id', 'rating', 'timestamp'])
users = pd.read_csv('./users.dat', sep='::', engine='python', names=['user_id', 'gender', 'age', 'occupation', 'zip'])

In [3]:
movies

,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanj(1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
3878,3948,Meet the Parents (2000),Comedy
3879,3949,Requiem for a Dream (2000),Drama
3880,3950,Tigerland (2000),Drama
3881,3951,Two Family House (2000),Drama


In [4]:
ratings

,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291
...,...,...,...,...
1000204,6040,1091,1,956716541
1000205,6040,1094,5,956704887
1000206,6040,562,5,956704746
1000207,6040,1096,4,956715648


In [5]:
users

,user_id,gender,age,occupation,zip
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455
...,...,...,...,...,...
6035,6036,F,25,15,32603
6036,6037,F,45,1,76006
6037,6038,F,56,1,14706
6038,6039,F,45,0,01060


In [6]:
merged_data = pd.merge(ratings, users, on='user_id')
merged_data = pd.merge(merged_data, movies, on='movie_id')

将3张表联立起来，充分利用所有的用户数据和物品数据

In [7]:
merged_data

,user_id,movie_id,rating,timestamp,gender,age,occupation,zip,title,genres
0,1,1193,5,978300760,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Animation|Children's|Musical
2,1,914,3,978301968,F,1,10,48067,My Fair Lady (1964),Musical|Romance
3,1,3408,4,978300275,F,1,10,48067,Erin Brockovich (2000),Drama
4,1,2355,5,978824291,F,1,10,48067,"Bug's Life, A (1998)",Animation|Children's|Comedy
...,...,...,...,...,...,...,...,...,...,...
1000204,6040,1091,1,956716541,M,25,6,11106,Weekend at Bernie's (1989),Comedy
1000205,6040,1094,5,956704887,M,25,6,11106,"Crying Game, The (1992)",Drama|Romance|War
1000206,6040,562,5,956704746,M,25,6,11106,Welcome to the Dollhouse (1995),Comedy|Drama
1000207,6040,1096,4,956715648,M,25,6,11106,Sophie's Choice (1982),Drama


这里选择了评分大于4的物品为正样本，否则为负样本

In [8]:
merged_data['label'] = (merged_data['rating'] >= 4).astype(int)

In [9]:
merged_data

,user_id,movie_id,rating,timestamp,gender,age,occupation,zip,title,genres,label
0,1,1193,5,978300760,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama,1
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Animation|Children's|Musical,0
2,1,914,3,978301968,F,1,10,48067,My Fair Lady (1964),Musical|Romance,0
3,1,3408,4,978300275,F,1,10,48067,Erin Brockovich (2000),Drama,1
4,1,2355,5,978824291,F,1,10,48067,"Bug's Life, A (1998)",Animation|Children's|Comedy,1
...,...,...,...,...,...,...,...,...,...,...,...
1000204,6040,1091,1,956716541,M,25,6,11106,Weekend at Bernie's (1989),Comedy,0
1000205,6040,1094,5,956704887,M,25,6,11106,"Crying Game, The (1992)",Drama|Romance|War,1
1000206,6040,562,5,956704746,M,25,6,11106,Welcome to the Dollhouse (1995),Comedy|Drama,1
1000207,6040,1096,4,956715648,M,25,6,11106,Sophie's Choice (1982),Drama,1


In [10]:
# 将用户和电影ID映射到连续的整数，与矩阵分解的操作一样
user_ids = merged_data['user_id'].unique()
movie_ids = merged_data['movie_id'].unique()

user_to_idx = {user_id: idx for idx, user_id in enumerate(user_ids)}
movie_to_idx = {movie_id: idx for idx, movie_id in enumerate(movie_ids)}

merged_data['user_idx'] = merged_data['user_id'].map(user_to_idx)
merged_data['movie_idx'] = merged_data['movie_id'].map(movie_to_idx)
merged_data

,user_id,movie_id,rating,timestamp,gender,age,occupation,zip,title,genres,label,user_idx,movie_idx
0,1,1193,5,978300760,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama,1,0,0
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Animation|Children's|Musical,0,0,1
2,1,914,3,978301968,F,1,10,48067,My Fair Lady (1964),Musical|Romance,0,0,2
3,1,3408,4,978300275,F,1,10,48067,Erin Brockovich (2000),Drama,1,0,3
4,1,2355,5,978824291,F,1,10,48067,"Bug's Life, A (1998)",Animation|Children's|Comedy,1,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1000204,6040,1091,1,956716541,M,25,6,11106,Weekend at Bernie's (1989),Comedy,0,6039,772
1000205,6040,1094,5,956704887,M,25,6,11106,"Crying Game, The (1992)",Drama|Romance|War,1,6039,1106
1000206,6040,562,5,956704746,M,25,6,11106,Welcome to the Dollhouse (1995),Comedy|Drama,1,6039,365
1000207,6040,1096,4,956715648,M,25,6,11106,Sophie's Choice (1982),Drama,1,6039,152


In [11]:
# 将性别编码为数值
gender_encoder = LabelEncoder()
merged_data['gender_encoded'] = gender_encoder.fit_transform(merged_data['gender'])
merged_data['gender_encoded']

0          0
1          0
2          0
3          0
4          0
          ..
1000204    1
1000205    1
1000206    1
1000207    1
1000208    1
Name: gender_encoded, Length: 1000209, dtype: int64

In [12]:
merged_data

,user_id,movie_id,rating,timestamp,gender,age,occupation,zip,title,genres,label,user_idx,movie_idx,gender_encoded
0,1,1193,5,978300760,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama,1,0,0,0
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Animation|Children's|Musical,0,0,1,0
2,1,914,3,978301968,F,1,10,48067,My Fair Lady (1964),Musical|Romance,0,0,2,0
3,1,3408,4,978300275,F,1,10,48067,Erin Brockovich (2000),Drama,1,0,3,0
4,1,2355,5,978824291,F,1,10,48067,"Bug's Life, A (1998)",Animation|Children's|Comedy,1,0,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1000204,6040,1091,1,956716541,M,25,6,11106,Weekend at Bernie's (1989),Comedy,0,6039,772,1
1000205,6040,1094,5,956704887,M,25,6,11106,"Crying Game, The (1992)",Drama|Romance|War,1,6039,1106,1
1000206,6040,562,5,956704746,M,25,6,11106,Welcome to the Dollhouse (1995),Comedy|Drama,1,6039,365,1
1000207,6040,1096,4,956715648,M,25,6,11106,Sophie's Choice (1982),Drama,1,6039,152,1


In [13]:
# 将电影类型编码进行multi-hot编码
genres = merged_data['genres'].str.get_dummies(sep='|')

In [14]:
genres

,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0
3,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
4,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1000204,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1000205,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0
1000206,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0
1000207,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0


In [15]:
# 提取特征和标签
X_user = merged_data[['user_idx', 'gender_encoded', 'age', 'occupation']].values
X_movie = merged_data[['movie_idx']].values
X_genres = genres.values
y = merged_data['label'].values

用户属性，分别是用户ID，用户性别，用户年龄，用户职业

In [16]:
print(X_user.shape)
X_user

(1000209, 4)


array([[   0,    0,    1,   10],
       [   0,    0,    1,   10],
       [   0,    0,    1,   10],
       ...,
       [6039,    1,   25,    6],
       [6039,    1,   25,    6],
       [6039,    1,   25,    6]])

电影的ID属性

In [17]:
print(X_movie.shape)
X_movie

(1000209, 1)


array([[  0],
       [  1],
       [  2],
       ...,
       [365],
       [152],
       [ 26]])

电影的分类属性，这里直接用了one-hot编码

In [18]:
print(X_genres.shape)
X_genres

(1000209, 18)


array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [19]:
print(y.shape)
y

(1000209,)


array([1, 0, 0, ..., 1, 1, 1])

In [20]:
# 将数据转换为PyTorch张量
X_user = torch.tensor(X_user, dtype=torch.float32)
X_movie = torch.tensor(X_movie, dtype=torch.long)
X_genres = torch.tensor(X_genres, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

print(X_user.shape)
print(X_user.shape)

# 创建数据集和数据加载器
dataset = TensorDataset(X_user, X_movie, X_genres, y)
dataloader = DataLoader(dataset, batch_size=1024, shuffle=True)

torch.Size([1000209, 4])
torch.Size([1000209, 4])


### 双塔模型建模与训练

torch.cat将每一层的特征连接了起来，形成了一个长条

In [21]:
class TwoTowerModel(nn.Module):
    def __init__(self, num_users, num_movies, embedding_dim, num_genres):
        super(TwoTowerModel, self).__init__()
        # 用户塔
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.user_fc = nn.Sequential(
            nn.Linear(embedding_dim + 3, 64),  # 用户ID + 性别 + 年龄 + 职业
            nn.ReLU(),
            nn.Linear(64, 32)
        )
        
        # 电影塔
        self.movie_embedding = nn.Embedding(num_movies, embedding_dim)
        self.movie_fc = nn.Sequential(
            nn.Linear(embedding_dim + num_genres, 64),  # 电影ID + 电影类型
            nn.ReLU(),
            nn.Linear(64, 32)
        )
        
        # 输出层，输出 1 个值（对应二分类）
        self.output_layer = nn.Linear(32, 1)
    
    def forward(self, X_user, X_movie, X_genres):
        
        # 用户嵌入
        user_idx = X_user[:, 0].long()
        user_embed = self.user_embedding(user_idx)
        user_embed = torch.cat([user_embed, X_user[:, 1:]], dim=1)
        user_embed = self.user_fc(user_embed)  # 形状: [batch_size, 32]
        
        # 电影嵌入
        movie_embed = self.movie_embedding(X_movie.squeeze())  # 形状: [batch_size, embedding_dim]
        movie_embed = torch.cat([movie_embed, X_genres], dim=1)  # 形状: [batch_size, embedding_dim + num_genres]
        movie_embed = self.movie_fc(movie_embed)  # 形状: [batch_size, 32]
        
        # 对用户嵌入和电影嵌入进行 L2 归一化
        user_embed_normalized = torch.nn.functional.normalize(user_embed, p=2, dim=1)
        movie_embed_normalized = torch.nn.functional.normalize(movie_embed, p=2, dim=1)
        
        # 计算余弦相似度
        cosine_similarity = (user_embed_normalized * movie_embed_normalized).sum(dim=1, keepdim=True)  # 形状: [batch_size, 1]
        
        # 应用 Sigmoid 函数，输出二分类结果
        output = torch.sigmoid(cosine_similarity)  # 形状: [batch_size, 1]
        return output.squeeze()  # 形状: [batch_size]


In [22]:
# 初始化模型、损失函数和优化器
num_users = len(user_ids)
num_movies = len(movie_ids)
embedding_dim = 8
num_genres = genres.shape[1]

model = TwoTowerModel(num_users, num_movies, embedding_dim, num_genres)
# criterion = nn.BCEWithLogitsLoss()  # 适用于二分类任务
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练模型，这里为了测试选了3轮，可适当调大维度
num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for X_user, X_movie, X_genres, y in dataloader:
        optimizer.zero_grad()
        outputs = model(X_user, X_movie, X_genres)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(dataloader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')

Epoch [1/3], Loss: 0.6363
Epoch [2/3], Loss: 0.5981
Epoch [3/3], Loss: 0.5823


这里用了交叉熵作损失函数，pytorch的交叉熵损失函数是通过nn.BCELoss()来调用的，对应TensorFlow的BinaryCrossEntropy</br>
交叉熵对于随机猜测的损失值为：−ln(0.5)≈0.693</br>
比0.693小，说明模型已经有所优化

### 为1号用户作推荐

四个特征为用户ID，用户性别，年龄（为保护隐私做了模糊处理），职业

In [23]:
# 获取1号用户的编码索引（假设user_id=1对应user_idx=0）
user_idx = 0  # 需确认merged_data中user_id=1对应的user_idx值
user_features = merged_data[merged_data['user_idx'] == user_idx].iloc[0]
X_user_sample = torch.tensor([[user_idx, 
                              user_features['gender_encoded'],
                              user_features['age'],
                              user_features['occupation']]], 
                              dtype=torch.float32)

X_user_sample

tensor([[ 0.,  0.,  1., 10.]])

In [24]:
# 获取用户已观看的电影（避免重复推荐）
watched_movies = merged_data[merged_data['user_idx'] == user_idx]['movie_idx'].tolist()

# 生成所有未观看的电影候选集
all_movie_indices = merged_data['movie_idx'].unique()
candidate_movies = [m for m in all_movie_indices if m not in watched_movies]

# 候选电影数
num_candidates = len(candidate_movies)

# 用户特征扩展（确保与候选电影数量一致）
# 这一步主要是为了做计算，需要算用户对每一部电影的打分，因此重复该用户的特征多次方便矩阵相乘
X_user_repeated = X_user_sample.repeat(num_candidates, 1)  # 形状 [num_candidates, 4]

# 电影特征构造（必须保持维度一致性）
X_movie_candidates = torch.tensor(candidate_movies, dtype=torch.long).unsqueeze(1)  # 形状 [num_candidates, 1]
X_genres_candidates = torch.tensor(genres.loc[candidate_movies].values, dtype=torch.float32)  # 形状 [num_candidates, 18]

print(X_movie_candidates)
print(X_genres_candidates)

tensor([[  53],
        [  54],
        [  55],
        ...,
        [3703],
        [3704],
        [3705]])
tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])


In [25]:
model.eval()
with torch.no_grad():
    # 重复用户特征以匹配候选电影数量
    X_user_repeated = X_user_sample.repeat(len(candidate_movies), 1)
    
    # 预测
    predictions = model(X_user_repeated, X_movie_candidates.unsqueeze(1), X_genres_candidates)
    probabilities = predictions.numpy()

In [26]:
probabilities

array([0.712002  , 0.7040161 , 0.67462564, ..., 0.6750569 , 0.64016443,
       0.72585434], dtype=float32)

In [27]:
# 将概率与电影ID关联
recommend_df = pd.DataFrame({
    'movie_idx': candidate_movies,
    'probability': probabilities
})

# 获取电影标题并排序
recommend_df = recommend_df.merge(merged_data[['movie_idx', 'title']].drop_duplicates(), on='movie_idx')
top_n_recommendations = recommend_df.sort_values('probability', ascending=False).head(10)

打印推荐列表

In [28]:
top_n_recommendations

,movie_idx,probability,title
314,367,0.729241,Nowhere (1997)
1015,1068,0.729196,On Her Majesty's Secret Service (1969)
1191,1244,0.729155,"Contender, The (2000)"
248,301,0.729040,Henry: Portrait of a Serial Killer (1990)
468,521,0.729024,"Wedding Singer, The (1998)"
1690,1743,0.729009,Halloween: H20 (1998)
633,686,0.728994,That Thing You Do! (1996)
1148,1201,0.728978,I Married A Strange Person (1997)
2353,2406,0.728835,Nightwatch (1997)
600,653,0.728791,Homeward Bound: The Incredible Journey (1993)
